> **교육 목표** 배깅과 랜덤포레스트가 나무 한 그루 대비 무엇을 좋게 하는지, 성능과 안정성 면에서 확인합니다.
>
> **핵심 내용** 나무 1그루 vs 배깅 vs 랜덤포레스트의 Train·Test F1 → 반복 실험으로 점수 흔들림 비교 → OOB 점수

In [ ]:
%%html
<style>
table {
    float: left;
    clear: both;
}
</style><!-- Markdown 왼쪽 정렬 -->

# D6_01 배깅과 랜덤포레스트

> **오늘 질문:** 나무를 여러 그루 모으면 무엇이 좋아질까?

3일차 Data Mart 피처로 고장을 예측합니다. 평가는 3일차와 같이 **시간 순 마지막 20%** 입니다.

| 표시 | 의미 |
|---|---|
| 📋 제공 | 코드가 들어 있습니다. 실행만 하세요 |
| ✍️ 입력 | 설명을 보고 직접 입력하세요 |
| 💬 해석 | 결과를 보고 답을 적어 보세요 |

## 0. 준비: 3일차 Data Mart 📋

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.metrics import f1_score

df = pd.read_parquet("../../data/factory_table.parquet")
df["temp_diff"] = df["process_temp"] - df["air_temp"]
df["power"] = df["torque"] * df["rpm"]
df["strain"] = df["tool_wear"] * df["torque"]
df["grade_num"] = df["grade"].map({"L": 0, "M": 1, "H": 2})
features = ["air_temp", "process_temp", "rpm", "torque", "tool_wear",
            "vibration", "temp_diff", "power", "strain", "grade_num"]

cut = int(len(df) * 0.8)
X_train, X_test = df[features].iloc[:cut], df[features].iloc[cut:]
y_train, y_test = df["failure"].iloc[:cut], df["failure"].iloc[cut:]

## 1. 나무 1그루 vs 배깅 vs 랜덤포레스트 ✍️

- **배깅**: 부트스트랩으로 뽑은 데이터마다 나무를 하나씩 (100그루)
- **랜덤포레스트**: 배깅 + 나눌 때마다 변수도 무작위로

> 변수 이름: `tree`, `bag`, `rf`

In [ ]:
# 힌트: BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, random_state=42)

## 2. Train · Test F1 비교 📋

In [ ]:
rows = []
for name, model in {"나무 1그루": tree, "배깅": bag, "랜덤포레스트": rf}.items():
    rows.append({"모델": name,
                 "Train F1": f1_score(y_train, model.predict(X_train)),
                 "Test F1": f1_score(y_test, model.predict(X_test))})
pd.DataFrame(rows).round(3)

## 3. 흔들림 실험 ✍️

Train에서 80%를 무작위로 10번 다르게 뽑아, 나무 1그루와 랜덤포레스트를 각각 학습합니다. 데이터가 조금 바뀔 때 Test F1이 얼마나 흔들리는지 봅니다. (20초 정도)

> 변수 이름: `scores`

In [ ]:
# 힌트: 표.sample(frac=0.8, random_state=seed).index 로 일부 행 고르기

## 4. 흔들림 그래프 ✍️

In [ ]:
# 힌트: 표.plot(kind="box")

## 5. OOB 점수 📋

부트스트랩에 뽑히지 않은 데이터(Out-Of-Bag)로, 따로 검증 데이터를 떼지 않고도 성능을 추정할 수 있습니다.

In [ ]:
rf_oob = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42).fit(X_train, y_train)
print(f"OOB 정확도 {rf_oob.oob_score_:.3f}  (불균형 데이터라 정확도는 참고만)")

## 정리 💬

**Q1.** 세 모델의 Train F1과 Test F1을 비교해 보세요. 이 데이터에서 배깅·랜덤포레스트가 나무 1그루보다 크게 나은가요?

> 답:


**Q2.** 흔들림 실험에서 두 모델의 표준편차와 최솟값·최댓값을 비교하면? 현장에서 왜 이 차이가 중요할까요?

> 답:
